# 34 — Cross-Attention: GROVER Global × ESM-2 Residues → pEC50

Fuses compound and protein representations via learned cross-attention:
- Query: projected GROVER-large global embedding (2400-dim → d_model) — graph topology
- Key/Value: per-residue ESM-2 embeddings (294 × 320) — PXR residues

The compound's global graph representation attends to individual protein residues,
producing a protein-conditioned molecule score. GROVER captures whole-molecule
topology while ESM-2 provides residue-level binding-site context.

Architecture: GROVER global (2400) → unsqueeze → single-token Q →
cross-attn over 294 protein residues → squeeze → FFN → pEC50

Complement to nb33 (ChemBERTa tokens × ESM-2): GROVER provides graph-topology
inductive bias vs. SMILES-sequence bias of ChemBERTa.

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import EsmModel, AutoTokenizer

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42; N_FOLDS = 5
torch.manual_seed(SEED)
DEVICE = 'cpu'
print(f'PyTorch {torch.__version__}  device={DEVICE}')

PyTorch 2.11.0+cpu  device=cpu


In [2]:
# ── 2. Load cached GROVER-large embeddings ────────────────────────────────────
GROVER_TR = DATA_PROCESSED / 'grover_large_train_emb.npy'
GROVER_TE = DATA_PROCESSED / 'grover_large_test_emb.npy'

if not GROVER_TR.exists():
    raise FileNotFoundError(f'Missing {GROVER_TR} — run nb22b first to generate GROVER-large embeddings')

grover_tr = torch.tensor(np.load(str(GROVER_TR)), dtype=torch.float32)  # (N_tr, 2400)
grover_te = torch.tensor(np.load(str(GROVER_TE)), dtype=torch.float32)  # (N_te, 2400)

print(f'GROVER-large train: {grover_tr.shape}  test: {grover_te.shape}')
D_GROVER = grover_tr.shape[1]  # 2400

GROVER-large train: torch.Size([4139, 2400])  test: torch.Size([513, 2400])


In [3]:
# ── 3. Pre-compute PXR residue embeddings (ESM-2 8M) ─────────────────────────
print('Loading ESM-2 (8M)...')
ESM_MODEL = 'facebook/esm2_t6_8M_UR50D'
esm_tok   = AutoTokenizer.from_pretrained(ESM_MODEL)
esm_enc   = EsmModel.from_pretrained(ESM_MODEL).eval()
for p in esm_enc.parameters(): p.requires_grad_(False)

D_PROT = 320

PXR_SEQ = ('GLTEEQRMMIRELMDAQMKTFDTTFSHFKNFRLPGVLSSGCELPESLQAPSREEAAKWSQVRKDLCS'
            'LKVSLQLRGEDGSVWNYKPPADSGGKEIFSLLPHMADMSTYMFKGIISFAKVISYFRDLPIEDQISL'
            'LKGAAFELCQLRFNTVFNAETGTWECGRLSYCLEDTAGGFQQLLLEPMLKFHYMLKKLQLHEEEYVL'
            'MQAISLFSPDRPGVLQHRVVDQLQEQFAITLKSYIECNRPQPAHRFLFLKIMAMLTELRSINAQHTQ'
            'RLLRIQDIHPFATPLMQELFGITGS')

with torch.no_grad():
    esm_in  = esm_tok(PXR_SEQ, return_tensors='pt')
    esm_out = esm_enc(**esm_in)
    pxr_residues = esm_out.last_hidden_state[0, 1:-1, :]  # (294, 320)

print(f'PXR residue embeddings: {pxr_residues.shape}')

Loading ESM-2 (8M)...


model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PXR residue embeddings: torch.Size([293, 320])


In [4]:
# ── 4. Cross-attention model ──────────────────────────────────────────────────
# GROVER produces a single global vector (not token sequence).
# We unsqueeze it to (B, 1, d_model) as a single-token query, then
# cross-attend over 294 protein residues → squeeze → predict.
# This learns: "which protein residues does this molecule's graph fingerprint
# most activate?"

class GROVERProtCrossAttn(nn.Module):
    def __init__(self, d_g=2400, d_p=320, d_model=256, n_heads=8, dropout=0.2):
        super().__init__()
        self.q_proj  = nn.Linear(d_g, d_model)
        self.kv_proj = nn.Linear(d_p, d_model)
        self.cross   = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(d_model)
        self.ff      = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 32),      nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, grover_emb, protein_residues):
        # grover_emb: (B, d_g)
        # protein_residues: (L_p, d_p)
        B = grover_emb.shape[0]
        q  = self.q_proj(grover_emb).unsqueeze(1)            # (B, 1, d_model)
        kv = self.kv_proj(protein_residues)                   # (L_p, d_model)
        kv = kv.unsqueeze(0).expand(B, -1, -1)               # (B, L_p, d_model)
        out, _ = self.cross(q, kv, kv)                       # (B, 1, d_model)
        out = self.norm(out.squeeze(1))                       # (B, d_model)
        return self.ff(out).squeeze(-1)                       # (B,)


class GROVERDataset(Dataset):
    def __init__(self, embs, labels):
        self.embs, self.labels = embs, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return self.embs[i], torch.tensor(self.labels[i], dtype=torch.float32)

print('Model architecture defined.')

Model architecture defined.


In [5]:
# ── 5. Training helpers ───────────────────────────────────────────────────────
def train_one_fold(embs_tr, y_tr, embs_va, y_va, pxr_res,
                   epochs=100, patience=15, lr=3e-4, bs=64):
    model = GROVERProtCrossAttn().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.MSELoss()

    ds_tr = GROVERDataset(embs_tr, y_tr)
    dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True)

    best_val, best_state, wait = float('inf'), None, 0
    for ep in range(epochs):
        model.train()
        for emb_b, y_b in dl_tr:
            emb_b, y_b = emb_b.to(DEVICE), y_b.to(DEVICE)
            opt.zero_grad()
            pred = model(emb_b, pxr_res)
            loss = criterion(pred, y_b)
            loss.backward(); opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(embs_va.to(DEVICE), pxr_res)
        val_loss = criterion(val_pred.cpu(), torch.tensor(y_va, dtype=torch.float32)).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = model(embs_va.to(DEVICE), pxr_res).cpu().numpy()
    return val_pred, model

print('Training helpers defined.')

Training helpers defined.


In [6]:
# ── 6. Scaffold 5-fold CV ──────────────────────────────────────────────────────
train = load_train()
te    = load_test()

y_tr      = train['pec50'].values
scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
pxr_res   = pxr_residues.to(DEVICE)

oof = np.full(len(y_tr), np.nan)
fold_raes = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    val_preds, _ = train_one_fold(
        grover_tr[tr_idx], y_tr[tr_idx],
        grover_tr[va_idx], y_tr[va_idx],
        pxr_res
    )
    oof[va_idx] = val_preds
    fold_rae = rae_fn(y_tr[va_idx], val_preds)
    fold_raes.append(fold_rae)
    print(f'  Fold {fold_i+1}: RAE={fold_rae:.4f}')

oof_rae = rae_fn(y_tr, oof)
print(f'\nOOF RAE: {oof_rae:.4f}  (fold mean: {np.mean(fold_raes):.4f} ± {np.std(fold_raes):.4f})')
print(f'GROVER-large standalone (nb22b):      0.6295')
print(f'ChemBERTa cross-attn (nb33):          see nb33')
print(f'GROVER cross-attn with ESM-2 (this):  {oof_rae:.4f}')

np.save(DATA_PROCESSED / 'oof_crossattn_grover_esm2.npy', oof)

  Fold 1: RAE=0.5607


  Fold 2: RAE=0.6321


  Fold 3: RAE=0.6493


  Fold 4: RAE=0.5812


  Fold 5: RAE=0.6697

OOF RAE: 0.6139  (fold mean: 0.6186 ± 0.0412)
GROVER-large standalone (nb22b):      0.6295
ChemBERTa cross-attn (nb33):          see nb33
GROVER cross-attn with ESM-2 (this):  0.6139


In [7]:
# ── 7. Full retrain + test predictions ────────────────────────────────────────
_, final_model = train_one_fold(
    grover_tr, y_tr,
    grover_tr[-200:], y_tr[-200:],
    pxr_res, epochs=120, patience=18
)

final_model.eval()
all_te_preds = []
BATCH = 128
with torch.no_grad():
    for i in range(0, len(grover_te), BATCH):
        p = final_model(
            grover_te[i:i+BATCH].to(DEVICE),
            pxr_res
        ).cpu().numpy()
        all_te_preds.append(p)

te_preds = np.concatenate(all_te_preds)
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED / 'te_crossattn_grover_esm2.npy', te_preds)

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'SMILES': te['smiles'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '34_crossattn_grover_esm2.csv'
sub.to_csv(out, index=False)
print(f'Saved: {out}  |  OOF RAE: {oof_rae:.4f}')
print(f'Test preds: mean={te_preds.mean():.3f}  std={te_preds.std():.3f}')

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\34_crossattn_grover_esm2.csv  |  OOF RAE: 0.6139
Test preds: mean=4.505  std=0.589
